# 실습: Groq API 기본 호출

Groq Python SDK와 Chat Completions API를 사용해 첫 호출부터 응답 구조 분석까지 진행합니다.


## 수업 진행 방식

- API 요청이 발생하는 셀은 한 번 실행한 뒤 결과를 확인합니다.
- Groq는 응답 속도가 빠르므로 스트리밍과 일반 호출의 체감 차이를 관찰합니다.
- 같은 셀을 여러 번 실행하면 사용량이 누적될 수 있습니다.
- 모든 API 키와 모델명은 `.env` 파일에서 읽습니다.


## 0. 준비 확인


In [1]:
import os
from pprint import pprint

import tiktoken
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(".env 파일에 GROQ_API_KEY를 먼저 설정하세요.")

MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")
client = Groq(api_key=api_key)

print(f"사용 모델: {MODEL}")


사용 모델: llama-3.1-8b-instant


## 1. 공통 도우미 함수


In [2]:
def print_usage(response) -> None:
    usage = getattr(response, "usage", None)
    if usage is None:
        print("토큰 사용량 정보가 없습니다.")
        return

    print("\n[토큰 사용량]")
    print(f"입력 토큰: {getattr(usage, 'prompt_tokens', None)}")
    print(f"출력 토큰: {getattr(usage, 'completion_tokens', None)}")
    print(f"총 토큰:   {getattr(usage, 'total_tokens', None)}")


def extract_text(response) -> str:
    return response.choices[0].message.content


def call_groq(label, request_func):
    print(f"\n=== {label} ===")
    try:
        response = request_func()
    except Exception as exc:
        status_code = getattr(exc, "status_code", None)
        print(f"Groq API 오류: {status_code} {type(exc).__name__}")
        print(exc)
        return None

    print(extract_text(response))
    print_usage(response)
    return response


## 2. 첫 번째 Groq API 호출


In [3]:
response = call_groq(
    "첫 번째 Groq API 호출",
    lambda: client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "Python을 처음 배우는 학생에게 변수의 의미를 한 문장으로 설명해줘.",
            }
        ],
        max_completion_tokens=180,
    ),
)



=== 첫 번째 Groq API 호출 ===
Python을 처음 배우는 학생에게는 다음과 같이 설명할 수 있습니다.

"변수란 어떤 값을 저장하기 위한 이름을 할당하는 것입니다. 쉽게 말하면, 변수는 값의 별명을 만들 수 있는 것이다. 예를 들어, "나의 이름"이라는 값을 저장하는 이름을 변수라고 하며, 그 이후로 "나의 이름"을 사용할 때 "변수명"이라고 하며 변수의 값을 사용할 수 있게 된다."

[토큰 사용량]
입력 토큰: 55
출력 토큰: 98
총 토큰:   153


## 3. 응답 구조 분석


In [4]:
response = call_groq(
    "응답 구조 분석용 호출",
    lambda: client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "딕셔너리를 초보자에게 설명하고, 짧은 예시 코드도 보여줘.",
            }
        ],
        max_completion_tokens=260,
    ),
)

if response is not None:
    print("\n[기본 필드]")
    print("id:", response.id)
    print("model:", response.model)
    print("created:", response.created)

    message = response.choices[0].message
    print("\n[message]")
    print("role:", message.role)
    print("content preview:", message.content[:120])

    print("\n[choice]")
    choice = response.choices[0]
    print("index:", choice.index)
    print("finish_reason:", choice.finish_reason)

    print("\n[usage 원본]")
    pprint(response.usage.model_dump() if hasattr(response.usage, "model_dump") else response.usage)



=== 응답 구조 분석용 호출 ===
딕셔너리는 key와 value를 가지는 자료형으로, dictionary라고도 불립니다. 딕셔너리는 리스트와는 다르게 중복된 key를 가질 수 없으며, key는 유일해야 합니다.

딕셔너리를 사용하는 이유는 key를 통해 value를 찾거나 값 변경이 가능하다는점이다. 이는 해시맵과 유사한 개념으로, key의 해시를 통해 위치를 찾는 방식으로 데이터를 저장합니다.

딕셔너리를 사용하는 예시 코드는 다음과 같습니다:

```python
# 딕셔너리 선언
person = {
    "이름": "홍길동",
    "나이": 25,
    "주소": "서울시 동대문구"
}

# key 조회
print(person["이름"])  # "홍길동" 출력

# key value 변경
person["나이"] = 30
print(person["나이"])  # 30 출력

# 새로운 key value 추가
person["직업"] = "개발자"
print(person)  # {"이름": "홍길동", "나이": 30, "주소": "서울시 동대문구", "직

[토큰 사용량]
입력 토큰: 58
출력 토큰: 260
총 토큰:   318

[기본 필드]
id: chatcmpl-f3c1da39-2a65-4198-8935-758c19881d95
model: llama-3.1-8b-instant
created: 1783055358

[message]
role: assistant
content preview: 딕셔너리는 key와 value를 가지는 자료형으로, dictionary라고도 불립니다. 딕셔너리는 리스트와는 다르게 중복된 key를 가질 수 없으며, key는 유일해야 합니다.

딕셔너리를 사용하는 이유는 key를 

[choice]
index: 0
finish_reason: length

[usage 원본]
{'completion_time': 0.421768007,
 'completion_tokens': 260,
 'completion_tokens_det

## 4. 시스템 메시지


In [5]:
role = "친절한 한국어 Python 선생님"

response = call_groq(
    "system 메시지로 역할 지정",
    lambda: client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    f"당신은 {role}입니다. "
                    "답변은 쉬운 비유 1개, 코드 1개, 확인 질문 1개로 구성하세요."
                ),
            },
            {
                "role": "user",
                "content": "리스트가 무엇인지 설명해줘.",
            },
        ],
        max_completion_tokens=360,
    ),
)



=== system 메시지로 역할 지정 ===
리스트는 집 같은 개념입니다. 집 안에는 물건을 많이 넣을 수 있고, 집 안에 있는 물건을 마음대로 빼거나 추가할 수 있습니다. 예를 들어, 학생들의 이름을 저장할 때 한 학생이 들어가고, 그 학생이 떠날 때 그 이름이 지워지거나 새로운 학생이 들어가면 새로운 이름이 추가됩니다.

```python
my_list = ['김학생', '유학생', '이학생']
# 학생들의 이름이 추가되었습니다.

my_list.append('박학생')
# 새로운 학생인 '박학생'이 들어갔습니다.

my_list.remove('유학생')
# 학생인 '유학생'이 집에서 나갔습니다.

print(my_list)
# ['김학생', '이학생', '박학생']이 출력됩니다.
```

이러한 리스트는 데이터를 관리하기 용이하고, 원하는 데이터를 쉽게 추가하거나 제거할 수 있다는 장점이 있습니다.

[토큰 사용량]
입력 토큰: 82
출력 토큰: 208
총 토큰:   290


### 역할 바꿔보기


In [6]:
role_examples = [
    "10살 학생에게 설명하는 선생님",
    "면접관처럼 핵심만 묻고 답하는 코치",
    "실무 개발자처럼 주의점을 강조하는 멘토",
]

for role in role_examples:
    call_groq(
        f"역할: {role}",
        lambda role=role: client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": f"당신은 {role}입니다. 답변은 5문장 이내로 작성하세요."},
                {"role": "user", "content": "파이썬 함수가 왜 필요한지 설명해줘."},
            ],
            max_completion_tokens=240,
        ),
    )



=== 역할: 10살 학생에게 설명하는 선생님 ===
함수(function)가 바로 코드를 짧고 쉽게 만드는 MAGIC TOOL이에요. 

예를 들어, 당신이 학교에서 일년 내내 공부하는 시간을 계산하는 함수를 만들고 싶은데, 매번 일시와 주차, 월을 직접 입력해야 한단말인가? 

그러기보다 일상에서 일년을 계산해주는 함수를 만들면, 한번만 만들면 된답니다. 

이러한 함수를 만드는게 파이썬에서 함수(function)의 가장 큰 장점이야!

[토큰 사용량]
입력 토큰: 76
출력 토큰: 118
총 토큰:   194

=== 역할: 면접관처럼 핵심만 묻고 답하는 코치 ===
함수의 개념을 이해해보겠습니다. 함수란 코드의 반복 실행을 줄였을 때 유용한 개념입니다. 

여러 줄의 반복되는 코드를 함수로 만든다면 코드의 길이와 복잡도를 줄일 수 있습니다. 

예를 들어, 10번이상 2의 제곱을 계산해야할 때 2**를 10번이나 입력해주기보다 더 간단하고 편리합니다.

[토큰 사용량]
입력 토큰: 81
출력 토큰: 100
총 토큰:   181

=== 역할: 실무 개발자처럼 주의점을 강조하는 멘토 ===
파이썬 함수의 역할은 프로그램의 로직을 더 나은 효율성과 재사용성으로 단순화하는 것입니다. 함수는 코드의 중복을 피하고, 유지 보수성을 높여줍니다. 만약 특정 로직을 여러 장소에서 호출해야 하는 경우, 함수를 사용하면 그 로직을 단 한 번만 작성해두고, 어디에서든 간에 쉽게 호출할 수 있습니다.

[토큰 사용량]
입력 토큰: 82
출력 토큰: 99
총 토큰:   181


## 5. 토큰 사전 추정


In [7]:
texts = [
    "안녕하세요.",
    "Python에서 리스트와 튜플의 차이를 설명해줘.",
    "초보자를 위한 함수 개념 설명과 예시 코드 2개를 작성해줘.",
]

try:
    encoding = tiktoken.encoding_for_model(MODEL)
except KeyError:
    encoding = tiktoken.get_encoding("cl100k_base")

for text in texts:
    token_count = len(encoding.encode(text))
    print(f"{token_count:>4} tokens | {text}")


   6 tokens | 안녕하세요.
  21 tokens | Python에서 리스트와 튜플의 차이를 설명해줘.
  27 tokens | 초보자를 위한 함수 개념 설명과 예시 코드 2개를 작성해줘.


## 6. 실제 사용량과 비교


In [8]:
prompt = "토큰을 아껴야 하는 이유를 Groq API 수업 관점에서 설명해줘."
estimated_tokens = len(encoding.encode(prompt))
print(f"호출 전 입력 토큰 추정: {estimated_tokens}")

response = call_groq(
    "토큰 사용량 비교",
    lambda: client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=180,
    ),
)


호출 전 입력 토큰 추정: 33

=== 토큰 사용량 비교 ===
토큰을 아껴야 하는 이유에 대해 Groq API 수업 관점에서 설명드리겠습니다.

**토큰이란?**

Groq API에서 토큰은 애플리케이션에서 사용자에 대한 식별 정보를 나타내는 문자열 데이터입니다. 토큰에는 사용자의 정보가 포함되고, 이 토큰을 사용하여 서버에 요청을 보낼 때 사용자 정보를 확인할 수 있습니다.

**토큰을 아껴야 하는 이유**

Groq API에서 토큰을 아껴야 하는 이유는 다음과 같습니다.

1. **사용자 정보 보호**: 사용자 정보를 포함하고 있는 토큰을 보호해야 합니다. 만약 토큰이 제 thirds 파티에 탈취되거나 누락된다

[토큰 사용량]
입력 토큰: 60
출력 토큰: 180
총 토큰:   240


## 7. 에러 처리


In [9]:
# 일부러 잘못된 API 키를 사용해 인증 오류를 관찰합니다.
# 실제 수업에서는 이 셀을 실행한 뒤 에러 메시지를 읽고 다음 셀로 넘어갑니다.

bad_client = Groq(api_key="gsk_invalid_key_for_class_demo")

bad_response = call_groq(
    "잘못된 API 키 테스트",
    lambda: bad_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "이 요청은 실패해야 정상입니다."}],
    ),
)



=== 잘못된 API 키 테스트 ===
Groq API 오류: 401 AuthenticationError
Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}


### 자주 만나는 오류

| 오류 | 확인할 것 |
| --- | --- |
| 401 Unauthorized | `.env`의 `GROQ_API_KEY` 값 |
| 403 Forbidden | 모델 또는 프로젝트 권한 |
| 413 Request Entity Too Large | 입력이 너무 길지 않은지 |
| 422 Unprocessable Entity | 입력 형식 또는 출력 형식 |
| 429 Too Many Requests | 짧은 시간에 너무 많이 호출했는지 |


## 8. 모델 목록 조회


In [10]:
models_response = None

try:
    models_response = client.models.list()
except Exception as exc:
    print("모델 목록 조회 실패:", type(exc).__name__, exc)

if models_response is not None:
    for model in models_response.data[:20]:
        print(model.id)


openai/gpt-oss-safeguard-20b
llama-3.3-70b-versatile
openai/gpt-oss-120b
canopylabs/orpheus-arabic-saudi
whisper-large-v3
allam-2-7b
meta-llama/llama-4-scout-17b-16e-instruct
groq/compound-mini
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
qwen/qwen3-32b
meta-llama/llama-prompt-guard-2-86m
llama-3.1-8b-instant
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
groq/compound
qwen/qwen3.6-27b
